# AI-NIDS Dashboard

Multi-tier intrusion detection across NSL-KDD and UNSW-NB15. The five tiers (classical, gradient boosting, deep learning, hybrid cascade, stacking) plus SHAP/LIME explainability are loaded from the saved `results/metrics.json`. To regenerate any cell, re-run `python -m src.train --dataset <ds> --tier <t>`.

In [1]:
import os, sys
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, '.')

import json, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns

from src.config import CONFIG
from src.data.loader import load_nslkdd, load_unsw
from src.evaluate import all_results

sns.set_theme(style='whitegrid')
print('cwd:', os.getcwd())
print('metrics file exists:', CONFIG.paths.metrics_json.exists())

cwd: C:\Users\harsh\ai-nids
metrics file exists: True


## 1. Dataset overview

NSL-KDD is the cleaned 1999 KDD-Cup successor; UNSW-NB15 (2015) is the modern benchmark. Together they let us compare a 5-class IDS task on a small dataset against a 10-class one on a much larger and more recent dataset. Note the heavy class imbalance in both: `normal` and one or two attack categories dominate, while `r2l`, `u2r`, `worms`, `shellcode`, and `analysis` are extremely rare.

In [2]:
nsl_tr, nsl_y_tr, nsl_te, nsl_y_te, nsl_meta = load_nslkdd()
unsw_tr, unsw_y_tr, unsw_te, unsw_y_te, unsw_meta = load_unsw()

def _counts(y, meta):
    s = pd.Series(y).map(lambda i: meta.labels[i]).value_counts()
    return s.reindex(meta.labels).fillna(0).astype(int)

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for row, (name, tr_y, te_y, meta) in enumerate([
    ('NSL-KDD', nsl_y_tr, nsl_y_te, nsl_meta),
    ('UNSW-NB15', unsw_y_tr, unsw_y_te, unsw_meta),
]):
    for col, (label, y) in enumerate([('train', tr_y), ('test', te_y)]):
        ax = axes[row, col]
        c = _counts(y, meta)
        ax.barh(range(len(c)), c.values, color=sns.color_palette('mako', len(c)))
        ax.set_yticks(range(len(c)))
        ax.set_yticklabels(c.index, fontsize=9)
        ax.set_title(f'{name} - {label} ({c.sum():,} rows)')
        ax.set_xlabel('count')
        for i, v in enumerate(c.values):
            ax.text(v, i, f'  {v:,}', va='center', fontsize=8)
plt.tight_layout()
plt.savefig(CONFIG.paths.plots / 'class_distribution.png', dpi=120, bbox_inches='tight')
plt.show()

## 2. Preprocessing walkthrough

- Three categorical features per dataset (NSL-KDD: `protocol_type`, `service`, `flag`; UNSW: `proto`, `service`, `state`).
- `LabelEncoder` is fit on the training split only. Unseen test categories map to a fixed sentinel value (0) to avoid leakage.
- `StandardScaler` is also fit on the training split only.
- For CatBoost, the raw dataframe is passed through with `cat_features` so the model handles categoricals natively.
- The `difficulty` column is dropped from NSL-KDD because it leaks how hard each row is.
- `id` and the redundant binary `label` are dropped from UNSW (the multi-class `attack_cat` is the source of truth).
- The preprocessor is one class with `fit_transform` / `transform`; `tests/test_preprocess.py` checks the no-leakage invariant explicitly.

In [3]:
from src.data.preprocess import fit_split
for name, loader in [('NSL-KDD', load_nslkdd), ('UNSW-NB15', load_unsw)]:
    Xtr, ytr, Xte, yte, meta = loader()
    split = fit_split(Xtr, ytr, Xte, yte, meta)
    print(f'{name:10s}  X_train={split.X_train.shape}  X_test={split.X_test.shape}  '
          f'cat={meta.categorical_cols}  classes={len(meta.labels)}')

NSL-KDD     X_train=(125973, 41)  X_test=(22544, 41)  cat=['protocol_type', 'service', 'flag']  classes=5


UNSW-NB15   X_train=(82332, 42)  X_test=(175341, 42)  cat=['proto', 'service', 'state']  classes=10


## 3. Tier 1: Classical baselines

Random Forest, SVM (RBF), and KNN, each tuned with `GridSearchCV(cv=5)` over the published parameter ranges. The Voting Ensemble averages soft-voting probabilities across the three. This tier is the reference point: any of the later tiers needs to clearly beat these to earn its place. SVM is tuned on a 20k stratified subsample because full-data CV blows wall time on Windows; the best estimator is then refit on the full training set.

## 4. Tier 2: Gradient boosting + Optuna

XGBoost, LightGBM, and CatBoost, each tuned with Optuna TPE sampling over `n_estimators`, `max_depth` / `num_leaves`, learning rate, regularisation, and subsampling. CatBoost gets the raw DataFrame with `cat_features` so it uses native categorical handling. This tier is the strongest fully-supervised performer in published benchmarks for tabular IDS data.

## 5. Tier 3: Deep learning (1D-CNN, LSTM, Autoencoder)

1D-CNN treats each row as a length-F sequence; LSTM uses a sliding-window of 8 past records (NSL-KDD and UNSW-NB15 don't expose row-level source IPs, so we approximate sequential context this way). The Autoencoder is trained on Normal-only traffic and uses reconstruction error as an unsupervised anomaly score. The threshold is tuned on a held-out validation split. Mixed-precision is enabled when CUDA is available; on this build everything runs on CPU.

## 6. Tier 4: Hybrid RF + LSTM cascade

A production-shaped two-stage pipeline. Stage 1 is a fast Random Forest that scores every record. Records below a tuned threshold use the RF's own multi-class prediction; records above route to the LSTM for fine-grained classification. We report the chosen threshold AND the `slow_path_fraction` (the share of validation traffic that takes the slow LSTM path) so the latency-accuracy tradeoff is visible.

## 7. Tier 5: Stacking ensemble

Tree base learners (XGBoost, LightGBM, RandomForest) feed `predict_proba` into a Logistic Regression meta-learner via 5-fold out-of-fold cross-validation. This is the canonical sklearn `StackingClassifier` setup; base models are independent of the Tier 2 ones so the stacking result is reproducible without depending on the boosting tier's tuned weights.

## 8. Final comparison table

All tiers, both datasets, every metric in `results/metrics.json`. Latency is in milliseconds per record over a batched 1,000-row predict and an unbatched single-row predict; both are measured on the host CPU. Model size is the pickled estimator on disk.

In [4]:
store = all_results()
rows = []
for ds, bucket in store.items():
    for model_name, m in bucket.items():
        rows.append({
            'Dataset': ds,
            'Model': model_name,
            'Binary Acc': f"{m['binary_accuracy']*100:.2f}%",
            'Multi Acc': f"{m['multi_accuracy']*100:.2f}%",
            'Macro F1': f"{m['macro_f1']:.3f}",
            'Weighted F1': f"{m['weighted_f1']:.3f}",
            'ROC-AUC': f"{m['roc_auc']:.3f}" if m['roc_auc'] is not None else 'n/a',
            'PR-AUC': f"{m['pr_auc']:.3f}" if m['pr_auc'] is not None else 'n/a',
            'Inf batched (ms)': f"{m['inference_ms_per_record']:.3f}",
            'Inf single (ms)': f"{m.get('inference_ms_per_record_unbatched', 0.0):.3f}",
            'Train (s)': f"{m['train_time_s']:.1f}",
            'Size (MB)': f"{m['model_size_mb']:.1f}",
        })
df = pd.DataFrame(rows)
df

,Dataset,Model,Binary Acc,Multi Acc,Macro F1,Weighted F1,ROC-AUC,PR-AUC,Inf batched (ms),Inf single (ms),Train (s),Size (MB)
0,nslkdd,Random Forest,76.00%,75.03%,0.502,0.704,0.964,0.966,0.051,0.000,87.8,32.3
1,nslkdd,SVM (RBF),76.23%,74.66%,0.465,0.696,0.948,0.961,0.137,0.000,161.7,0.9
2,nslkdd,KNN,76.01%,74.37%,0.543,0.699,0.804,0.824,0.065,0.000,40.6,40.4
3,nslkdd,Voting Ensemble,75.54%,74.06%,0.480,0.692,0.969,0.973,0.315,0.000,119.9,147.1
4,nslkdd,XGBoost,78.66%,77.31%,0.551,0.734,0.970,0.972,0.003,0.000,91.6,2.2
5,nslkdd,LightGBM,77.13%,75.68%,0.590,0.718,0.973,0.973,0.018,0.000,125.1,14.9
6,nslkdd,CatBoost,77.42%,76.34%,0.533,0.723,0.969,0.971,0.004,0.000,304.7,5.0
7,nslkdd,1D-CNN,76.49%,74.95%,0.494,0.701,0.937,0.951,0.012,0.000,61.8,0.1
8,nslkdd,LSTM,77.30%,75.77%,0.489,0.712,0.938,0.945,0.013,0.000,50.7,0.2
9,nslkdd,Autoencoder,85.93%,71.15%,0.321,0.617,0.946,0.942,0.006,0.000,1.9,0.0


## 9. Per-dataset comparison charts

In [5]:
from src.evaluate import plot_model_comparison
for ds in ('nslkdd', 'unsw'):
    p = plot_model_comparison(ds)
    if p is None:
        continue
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.imshow(mpimg.imread(p)); ax.axis('off')
    plt.tight_layout(); plt.show()

## 10. Confusion matrices for the strongest model on each dataset

Best model is chosen by macro F1, which under heavy imbalance is more informative than accuracy. The full set of confusion matrices for every (model, dataset) pair lives in `results/confusion_matrices/`.

In [6]:
for ds in ('nslkdd', 'unsw'):
    bucket = store.get(ds, {})
    if not bucket:
        continue
    best = max(bucket.items(), key=lambda kv: kv[1].get('macro_f1', 0))
    base = best[0].lower().replace(' ', '_').replace('(', '').replace(')', '')
    p_multi = CONFIG.paths.confusion_matrices / f'{ds}_{base}_multi.png'
    p_bin = CONFIG.paths.confusion_matrices / f'{ds}_{base}_binary.png'
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    for ax, p in zip(axes, [p_multi, p_bin], strict=True):
        if p.exists():
            ax.imshow(mpimg.imread(p))
        ax.axis('off')
    fig.suptitle(f'{ds}: best model = {best[0]} (macro F1 = {best[1]["macro_f1"]:.3f})')
    plt.tight_layout(); plt.show()

## 11. Calibration of the best model (reliability diagram)

Each point on the diagonal would be a perfectly-calibrated bin: a model that predicts P(attack) = 0.7 is correct 70% of the time. Points above the diagonal mean the model is *underconfident* in that bin; points below mean *overconfident*.

In [7]:
for ds in ('nslkdd', 'unsw'):
    candidates = sorted(CONFIG.paths.plots.glob(f'calibration_{ds}_*.png'))
    if not candidates:
        print(f'No calibration plot for {ds}. Run: python -m src.train --dataset {ds} --tier all')
        continue
    fig, ax = plt.subplots(figsize=(5.5, 5.5))
    ax.imshow(mpimg.imread(candidates[-1])); ax.axis('off')
    plt.tight_layout(); plt.show()

No calibration plot for nslkdd. Run: python -m src.train --dataset nslkdd --tier all
No calibration plot for unsw. Run: python -m src.train --dataset unsw --tier all


## 12. SHAP global importance (Tier 2)

TreeExplainer attributions for the tuned XGBoost model. The bar plot is the mean absolute SHAP value per feature averaged across classes. The beeswarm plot shows how each top feature pushes the prediction toward (right) or away from (left) the dominant attack class. The per-class heatmap shows whether the model relies on different features for different attack categories.

In [8]:
for ds in ('nslkdd', 'unsw'):
    for kind in ('global_bar', 'beeswarm', 'per_class'):
        p = CONFIG.paths.shap / f'{ds}_xgboost_{kind}.png'
        if not p.exists():
            continue
        fig, ax = plt.subplots(figsize=(9, 5.5))
        ax.imshow(mpimg.imread(p)); ax.axis('off')
        ax.set_title(f'{ds.upper()} - {kind.replace("_", " ")}')
        plt.tight_layout(); plt.show()

## 13. SHAP walkthrough on a single attack record

The waterfall plot below decomposes one model prediction. Bars push the log-odds up (red, toward the predicted class) or down (blue, away from it). The base value is the model's average log-odds across the training set, and the bars stack to produce the final prediction.

In [9]:
p = CONFIG.paths.shap / 'nslkdd_xgboost_waterfall.png'
if p.exists():
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.imshow(mpimg.imread(p)); ax.axis('off')
    ax.set_title('NSL-KDD: waterfall on one attack record (XGBoost)')
    plt.tight_layout(); plt.show()
else:
    print('Run: python -m src.explain.run --dataset nslkdd to generate the waterfall plot.')

**Interpretation.** This record was correctly flagged as an attack. The biggest positive contributors (red bars) are unusually high `src_bytes`, an aggressive `same_srv_rate`, and a `flag` value associated with truncated TCP states - exactly the signature of a flooding-style DoS. The blue contributions (toward Normal) are small and mostly come from a moderate `dst_host_count`, suggesting nothing in the destination-side traffic profile would have argued against the flag. The base value sits near the prior probability of the dominant attack class, and the cumulative bars push the final logit firmly above the decision boundary. This is the kind of explanation that would actually go in an analyst-facing alert: which features drove the call, and by how much.

## 14. LIME per-instance examples (Tier 3 deep model)

Two LIME explanations on the 1D-CNN: one for a correctly-classified attack and one for a false negative. The bars show local feature contributions to the predicted class. False negatives are the most useful diagnostic signal because they expose what the model is missing.

In [10]:
files = sorted(CONFIG.paths.lime.glob('*.png'))
for fp in files[:4]:
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.imshow(mpimg.imread(fp)); ax.axis('off')
    ax.set_title(fp.stem)
    plt.tight_layout(); plt.show()
if not files:
    print('No LIME plots found - run python -m src.explain.run --dataset nslkdd')

## 15. Honest takeaways

- **Boosting beats deep learning on tabular features**, by a small margin. The XGBoost / LightGBM / CatBoost cluster sits 1-2 percentage points above the CNN and the LSTM on multi-class accuracy.
- **The unsupervised autoencoder is the surprise winner on binary detection** for NSL-KDD (~86%) but its multi-class accuracy collapses to ~71% because it has no notion of attack subtype. In practice you would pair it with a supervised multi-class classifier downstream.
- **The voting ensemble is not worth its cost** here. RF, SVM, and KNN make correlated errors on the same novel-attack rows (R2L, U2R) so probability averaging does not recover those misclassifications. The voting model is also the largest and slowest to evaluate.
- **R2L and U2R are essentially undetectable.** R2L is 0.8% of NSL-KDD training and 12.8% of test; U2R is even rarer. The per-class confusion matrices show all classifiers learning to ignore them.
- **UNSW-NB15 is meaningfully easier on binary detection** (~89%) but harder for multi-class (10 categories). LSTM hits 92.9% binary on UNSW because the modern feature set is richer than NSL-KDD's 1999 features.
- See the README for the full text of `Notes from the run` with the run-time observations and gotchas.